# Выбор разбиения по ограниченной внешней информации
Что меняется, если при малом числе меток разрешить внутреннему индексу выбрать альтернативу внешнему лидеру? Пул разбиений строится по признакам. Часть объектов выделяется для проверки и все правила сравниваются на одних и тех же скрытых метках.

**M1** максимизирует скорректированный индекс Рэнда (ARI) на открытых метках. **M2** использует внутренний индекс при равенстве ARI. **M5** допускает отставание до 0,05 ARI. **M6** допускает отставание в пределах одного стандартного отклонения парной бутстрепной разницы ARI между исходным лидером и кандидатом. Во всех перевыборках сравниваем кандидатов на одних объектах. Среди допущенных побеждает максимум silhouette (в архивном опыте Digits DB — минимум индекса Дэвиса — Болдина). Выигрыш — проверочный ARI решения M6 минус проверочный ARI решения M1.

## Окружение и параметры
Из папки `Diploma/code` один раз выполните `python -m pip install -e '.[text]' jupyterlab`, затем `python -m jupyterlab notebooks/light_experiments.ipynb`. Для CUDA выберите ядро с CuPy, cuML со спектральной кластеризацией и PyTorch с поддержкой CUDA. На Mac используйте CPU. Архивные разделы читают распакованную папку с подпапками `results`, `artifacts` и `data`. При переносе укажите её путь в ARCHIVE. Каждый раздел запускается независимо после настройки.

In [ ]:
from pathlib import Path
from time import perf_counter
import sys
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"axes.grid": True, "axes.axisbelow": True, "grid.alpha": 0.35, "grid.linestyle": ":"})
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/clusterization").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from clusterization.light_series import DOMAINS, archived, domain, rectangles

DEVICE = "cpu"  # "cuda" для NVIDIA
ARCHIVE = ROOT.parents[1] / "outputs/oslide/replay_source"
DATASETS = list(DOMAINS)  # например ["digits"] для короткого запуска раздела 2
OUT = ROOT / "reports" / "light_series" / DEVICE
OUT.mkdir(parents=True, exist_ok=True)

def measure(name, fn, *args):
    start = perf_counter()
    result = fn(*args)
    print(f"{name}: {perf_counter() - start:.3f} с")
    return result

def save(frame, name):
    frame.to_csv(OUT / f"{name}.csv", index=False)
    return frame

## 1. Где теряется качество в архивных пилотах?
Четыре сохранённых прогона. При каждом раскрытии меток восстанавливаем четыре решения, сверяем их с архивом и разбиваем потерю на две части. Первая — насколько лучший допущенный уступает лучшему во всём пуле. Вторая — насколько выбранный уступает лучшему допущенному. Таблица хранит идентификаторы, алгоритмы и число кластеров для всех четырёх правил.

In [ ]:
pilot = save(measure("Архив", archived, ARCHIVE, DEVICE), "pilots")
pilot.groupby("pilot")[["excluded_loss", "inside_loss"]].mean().rename(columns={
    "excluded_loss":"Потеря при допуске", "inside_loss":"Потеря внутри допуска"
}).plot.barh(stacked=True, xlabel="Средняя потеря проверочного ARI", ylabel="")
plt.tight_layout()
display(pilot.loc[pilot.gain < -1e-12, ["pilot","scenario","repeat","budget","M1_candidate","M2_candidate","M5_candidate","M6_candidate","gain"]])
digits = pilot[pilot.pilot.str.contains("digits")].pivot(index=["scenario","repeat","budget"], columns="pilot", values="M6")
display(digits.assign(same_choice=digits.iloc[:,0] == digits.iloc[:,1]))

## 2. Меняется ли выигрыш между наборами данных?
До 600 объектов, 20 повторов, вложенные бюджеты 20 и 50 меток, 100 бутстрепных выборок. Половина групп выделена для проверки. Числовые признаки и яркости стандартизируем. Для текстов используем нормированные MiniLM-эмбеддинги и 50 главных компонент. Строим KMeans с 2, 4, 8 и 16 кластерами и спектральные разбиения с 4 и 8 кластерами. Silhouette использует евклидовы расстояния в том же пространстве. Таблица сохраняется после каждого набора. Первая загрузка текстов и модели требует интернета.

In [ ]:
frames, timings = [], []
for name in DATASETS:
    frame, timing = measure(name, domain, ROOT, name, DEVICE)
    frames.append(save(frame, name))
    timings.append(timing)
    save(pd.DataFrame(timings), "domain_times")
domains = pd.concat(frames, ignore_index=True)
summary = domains.groupby(["domain","dataset","budget"]).agg(
    gain=("gain","mean"), switches=("switched","sum"),
    worse=("gain",lambda values: (values < -1e-12).sum()), coverage=("coverage","mean"),
    candidates=("pool_size","first"), min_ari=("pool_min","first"), best_ari=("oracle","first")
).reset_index()
display(save(summary, "domain_summary"))
display(summary.groupby(["domain","budget"]).gain.mean().rename("Среднее с равным весом наборов"))
display(pd.DataFrame(timings))

In [ ]:
fig, axes = plt.subplots(1, 2, sharex=True, sharey=True, figsize=(10, 4))
colors = {"измерения":"C0", "тексты":"C1", "изображения":"C2"}
for ax, (budget, block) in zip(axes, summary.groupby("budget")):
    for group, rows in block.groupby("domain"):
        ax.scatter(rows.gain, rows.dataset, color=colors[group], label=group)
    ax.axvline(0, color="grey", lw=1)
    ax.set(title=f"{budget} меток", xlabel="Средний выигрыш M6 по ARI")
axes[-1].legend()
fig.tight_layout()

## 3. Как смысл класса влияет на выбор при одной геометрии?
Три реализации прямоугольника, по 100 точек в каждой из четырёх вершин, шум 0,2. Сравниваем метки «слева / справа» и «сверху / снизу». Координаты, пул, скрытые объекты, раскрытые индексы и бутстрепные веса одинаковы внутри пары. В пуле KMeans с 2, 3 и 4 кластерами и оба контрольных разбиения. Бюджеты 4, 8 и 20, по 20 повторов и 100 перевыборок. Столбец controls_tied_on_labels показывает, задают ли два контроля одинаковое разбиение размеченных точек.

In [ ]:
rect = save(measure("Прямоугольник", rectangles, DEVICE), "rectangles")
paired = rect.pivot(index=["seed","repeat","budget"], columns="target", values="gain")
paired["difference"] = paired["left-right"] - paired["up-down"]
display(paired.groupby("budget").mean())
display(rect.groupby(["target","budget"])[["control_silhouette","both_classes","controls_tied_on_labels","switched"]].mean())
paired.groupby("budget")[["left-right","up-down"]].mean().plot.bar(
    ylabel="Средний выигрыш M6 по ARI", xlabel="Число открытых меток", rot=0)

## 4. Устойчив ли выбор к случайности самого бутстрепа?
Берём первые по бюджету и номеру повторы случайной разметки Digits и Breast Cancer с переключением и без него, если доступны. На тех же метках повторяем M6 с 30, 100 и 300 перевыборками и пятью начальными значениями генератора. Смотрим число разных победителей, размер допуска и долю совпадений с архивным выбором. Ближайший к порогу кандидат записан в CSV вместе с запасом допуска (положительное значение означает прохождение).

In [ ]:
stability = save(measure("Бутстреп", archived, ARCHIVE, DEVICE, True), "bootstrap")
st = stability.groupby(["pilot","budget","repeat","archived_switch","draws"]).agg(
    winners=("M6","nunique"), min_eligible=("eligible","min"),
    max_eligible=("eligible","max"), same=("same_as_archive","mean"))
display(st)
st["same"].unstack("draws").plot.bar(ylabel="Доля совпадений с архивом", ylim=(0,1.05),
    xlabel="Пилот, бюджет, повтор, исходное переключение", rot=20)
plt.tight_layout()

## Дополнительная диагностика по обзору
Абсолютные потери, M0/M7, вложенные пулы, состав классов, пороги и устойчивость бутстрепа.

In [ ]:
import subprocess
diagnostics = ROOT / "reports/review_recomputed"
refinement = ROOT / "reports/review_updated_recomputed"
subprocess.run([sys.executable, str(ROOT / "scripts/review_analysis.py"), "--root", str(ROOT), "--out", str(diagnostics)], check=True)
subprocess.run([sys.executable, str(ROOT / "scripts/refine_review_analysis.py"), "--results", str(diagnostics), "--out", str(refinement)], check=True)
subprocess.run([sys.executable, str(ROOT / "scripts/build_experiment_figures.py"), "--results", str(ROOT / "reports/light_series/cpu"), "--diagnostics", str(diagnostics), "--refinement", str(refinement), "--out", str(ROOT.parent / "paper/figures")], check=True)
